# Transform properties

The full polar Fourier transform (PFT) has its own catalogue of analytical properties -- facts that hold regardless of what signal is being transformed -- from the first of Baddour's two-part polar-coordinates paper [[1]](#ref-1). The discrete Hankel transform's own properties are covered in [03_discrete_hankel_transform](03_discrete_hankel_transform.ipynb). This notebook is a tour of a representative handful of the PFT's -- the full catalogue is machine-checked in this package's own test suite, not just illustrated here.

In [ ]:
import numpy as np

import pypft


## PFT kernel orthogonality and the delta pair

The PFT's own combined kernel `E^-`/`E^+` obeys the same two flavors of
discrete orthogonality [[1]](#ref-1) (Eqs. 34, 37) that already underlie
every round trip in the previous notebook, plus its own version of the
Kronecker-delta pair: transforming `E^+`'s own column at a fixed frequency
index reproduces a delta at exactly that index [[1]](#ref-1) (Eq. 43).
`pypft._kernel` builds that combined kernel explicitly and from first
principles as this package's own test oracle -- but `inverse_pft` applied to
a standard basis array *is* that same column, so the public pipeline alone
already demonstrates the property operationally:

In [ ]:
grid = pypft.PolarGrid(n_radial=8, n_angular=5, R=3.0)

basis = np.zeros((grid.n_radial, grid.n_angular), dtype=complex)
q0, m0 = 2, 3
basis[m0, q0] = 1.0

e_plus_column = pypft.inverse_pft(basis, grid)
recovered = pypft.forward_pft(e_plus_column, grid)
float(np.abs(recovered - basis).max())


## Rotation equivariance

Circularly shifting the frequency-domain array by `q0` spokes and then
inverse-transforming gives exactly the same result as inverse-transforming
first and *then* shifting [[1]](#ref-1) (Eq. 75) -- rotating the transform is
the same as rotating the underlying function:

In [ ]:
rng = np.random.default_rng(1)
F_grid = rng.standard_normal((grid.n_radial, grid.n_angular)) + 1j * rng.standard_normal(
    (grid.n_radial, grid.n_angular)
)
f_grid = pypft.inverse_pft(F_grid, grid)

q0 = 2
lhs = pypft.inverse_pft(np.roll(F_grid, q0, axis=1), grid)
rhs = np.roll(f_grid, q0, axis=1)
float(np.abs(lhs - rhs).max())


## Linearity and the DC term

`forward_pft`/`inverse_pft` are linear maps, like every Fourier-family
transform, and a function with no angular dependence at all reduces the
whole pipeline to a single order-0 Hankel transform (scaled by `2*pi`, the
harmonic-0 term's own scale factor) -- the PFT's analogue of a continuous 2-D
Fourier transform's DC/mean term:

In [ ]:
radial_profile = rng.standard_normal(grid.n_radial)
f_symmetric = np.broadcast_to(radial_profile[:, np.newaxis], f_grid.shape)

F_symmetric = pypft.forward_pft(f_symmetric, grid)
expected_column = 2.0 * np.pi * pypft.hankel_transform(radial_profile, 0, grid.R)
float(np.abs(F_symmetric - expected_column[:, np.newaxis]).max())


## Where to go next

The transform's properties are what let the rest of this package be built
with confidence: typed domains, batching, and visualization all lean on the
pipeline behaving exactly as these identities say it must. The next notebook
introduces the typed domain objects that track where in the chain a signal
currently sits.

## References

1. <a id="ref-1"></a> N. Baddour, "Discrete Two-Dimensional Fourier Transform in Polar Coordinates Part I: Theory and Operational Rules," *Mathematics*, 7(8):698, 2019. [doi:10.3390/math7080698](https://doi.org/10.3390/math7080698)